# Inspect Dense LeJEPA checkpoints (CPU inference)

Use this after `examples/dense_lejepa_ddp_spawn.py` (or any run that produced `checkpoint_*.pt`).

- Pick a **`checkpoint_latest.pt`** (or a specific `checkpoint_epoch_####.pt`).
- Load weights on **CPU** and run a short forward on synthetic disc-square images (same as training) or your own grayscale files.
- Visualize **per-hook** dense latents (L2 norm across channels, view 0).

**Tip:** Training often uses `latent.step_mode="rotate"`. Here we temporarily set **`joint`** so one forward fills every key in `latents_by_source`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

# Repo root (parent of examples/ if you launch from examples/)
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "examples":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from examples.dense_lejepa_ddp_spawn import load_dense_lejepa_from_checkpoint
from local_conv_attention import DiscSquareDataset

device = torch.device("cpu")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("device:", device)

In [ ]:
# --- user settings ---
OUTPUT_ROOT = PROJECT_ROOT / "examples" / "dense_lejepa_ddp_outputs"
# Set to a file, or None to auto-pick newest run's checkpoint_latest.pt
CHECKPOINT_PATH: Path | None = None  # e.g. OUTPUT_ROOT / "20260319_120000" / "checkpoint_latest.pt"

NUM_IMAGES = 4  # synthetic samples
IMAGE_SIZE = 128  # training default for spawn demo; change if your checkpoint used another size

# Optional: grayscale image paths (PNG/JPG). Resized to IMAGE_SIZE x IMAGE_SIZE
CUSTOM_IMAGE_PATHS: list[Path] = []

In [ ]:
def find_default_checkpoint(root: Path) -> Path | None:
    if not root.is_dir():
        return None
    runs = sorted([p for p in root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
    for run in runs:
        cand = run / "checkpoint_latest.pt"
        if cand.is_file():
            return cand.resolve()
    return None


def list_checkpoints(root: Path) -> list[Path]:
    if not root.is_dir():
        return []
    files: list[Path] = []
    for run in sorted(root.iterdir()):
        if not run.is_dir():
            continue
        files.extend(sorted(run.glob("checkpoint_epoch_*.pt")))
        latest = run / "checkpoint_latest.pt"
        if latest.is_file():
            files.append(latest.resolve())
    return files


print("Known runs under", OUTPUT_ROOT)
for p in sorted([d for d in OUTPUT_ROOT.glob("*") if d.is_dir()], key=lambda x: x.stat().st_mtime, reverse=True)[:8]:
    print(" ", p.name, "| latest:", (p / "checkpoint_latest.pt").is_file())

ckpt_path = Path(CHECKPOINT_PATH).resolve() if CHECKPOINT_PATH else find_default_checkpoint(OUTPUT_ROOT)
if ckpt_path is None or not ckpt_path.is_file():
    raise FileNotFoundError(
        f"No checkpoint found. Set CHECKPOINT_PATH or train with dense_lejepa_ddp_spawn.py. "
        f"Tried OUTPUT_ROOT={OUTPUT_ROOT}"
    )
print("Using checkpoint:", ckpt_path)

In [ ]:
try:
    raw = torch.load(ckpt_path, map_location="cpu", weights_only=False)
except TypeError:
    raw = torch.load(ckpt_path, map_location="cpu")
print("Checkpoint keys:", sorted(raw.keys()))
for k in ("epoch", "global_step"):
    if k in raw:
        print(f"  {k}:", raw[k])

model, experiment_cfg = load_dense_lejepa_from_checkpoint(ckpt_path, map_location=device)
model.eval()
print("Model on device:", next(model.parameters()).device)
print("latent.step_mode (from cfg):", model.config.latent.step_mode)
print("latent hooks:", model.config.latent.resolved_sources())

cfg_json = ckpt_path.parent / "config.json"
if cfg_json.is_file():
    print("config.json present (first keys):", list(json.loads(cfg_json.read_text()).keys())[:6])

In [ ]:
def grayscale_paths_to_batch(paths: list[Path], size: int) -> torch.Tensor:
    from PIL import Image

    tiles: list[torch.Tensor] = []
    for p in paths:
        im = Image.open(p).convert("L")
        try:
            resample = Image.Resampling.BILINEAR
        except AttributeError:
            resample = Image.BILINEAR
        im = im.resize((size, size), resample)
        arr = np.asarray(im, dtype=np.float32) / 255.0
        tiles.append(torch.from_numpy(arr).unsqueeze(0))
    return torch.stack(tiles, dim=0)


in_ch = experiment_cfg.model.in_channels
tiles: list[torch.Tensor] = []

ds = DiscSquareDataset(repeats_per_type=max(1, NUM_IMAGES // 8 + 1), image_size=IMAGE_SIZE)
for i in range(NUM_IMAGES):
    tiles.append(ds[i]["image"])
synthetic = torch.stack(tiles[:NUM_IMAGES], dim=0)

custom = None
if CUSTOM_IMAGE_PATHS:
    custom = grayscale_paths_to_batch([Path(p).resolve() for p in CUSTOM_IMAGE_PATHS], IMAGE_SIZE)

batch_cpu = synthetic.to(dtype=torch.float32)
if custom is not None:
    batch_cpu = torch.cat([batch_cpu, custom], dim=0)

if batch_cpu.shape[1] != in_ch:
    raise ValueError(f"Images have {batch_cpu.shape[1]} channels; model expects in_channels={in_ch}")

print("batch shape:", tuple(batch_cpu.shape))  # B,1,H,W

In [ ]:
# One forward with all hooks (joint). Restore training mode after.
_prev_mode = model.config.latent.step_mode
model.config.latent.step_mode = "joint"
with torch.no_grad():
    outputs = model(batch_cpu.to(device), rotate_latent_index=0)
model.config.latent.step_mode = _prev_mode

lat_by_src = outputs["latents_by_source"]
print("Output keys:", outputs.keys())
print("latents (primary) shape:", tuple(outputs["latents"].shape))
for name, t in lat_by_src.items():
    print(f"  {name}: {tuple(t.shape)}  # B, V, D, H, W")
print("diagnostic losses (not used at inference):", float(outputs["loss"]), float(outputs["inv_loss"]))

In [ ]:
def latent_norm_map(lat_bvdhw: torch.Tensor, b: int, v: int = 0) -> np.ndarray:
    """L2 norm over channel dim -> HxW heatmap."""
    t = lat_bvdhw[b, v].detach().float().cpu()
    m = torch.linalg.vector_norm(t, dim=0).numpy()
    m = m - m.min()
    if m.max() > 0:
        m = m / m.max()
    return m


def show_inputs(batch: torch.Tensor, title: str = "inputs") -> None:
    b = batch.size(0)
    fig, axes = plt.subplots(1, b, figsize=(2.5 * b, 2.5))
    if b == 1:
        axes = [axes]
    for i in range(b):
        axes[i].imshow(batch[i, 0].cpu().numpy(), cmap="gray", vmin=0.0, vmax=1.0)
        axes[i].axis("off")
        axes[i].set_title(f"#{i}")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


sources = list(lat_by_src.keys())
n_show = min(batch_cpu.size(0), 6)
fig_h = 2.3 * n_show
fig_w = 2.2 * (1 + len(sources))
fig, axes = plt.subplots(n_show, 1 + len(sources), figsize=(fig_w, fig_h))
if n_show == 1:
    axes = axes[None, :]
for r in range(n_show):
    axes[r, 0].imshow(batch_cpu[r, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    axes[r, 0].axis("off")
    axes[r, 0].set_ylabel(f"img {r}", rotation=90, labelpad=8)
    if r == 0:
        axes[r, 0].set_title("input")
    for j, name in enumerate(sources, start=1):
        axes[r, j].imshow(latent_norm_map(lat_by_src[name], r), cmap="viridis", vmin=0, vmax=1)
        axes[r, j].axis("off")
        if r == 0:
            axes[r, j].set_title(name.replace("_", " "), fontsize=8)
plt.suptitle("Dense latent L2-norm maps (view 0)", y=1.02)
plt.tight_layout()
plt.show()

### Notes

- **Input size:** The spawn demo uses `128×128`. If you change `IMAGE_SIZE`, ensure spatial dimensions still work with the HEA stride pyramid (use multiples of 32 or the size you trained with).
- **All hooks:** We set `model.config.latent.step_mode = "joint"` only for the forward above; your checkpoint on disk is unchanged.
- **Rotate mode:** To mimic training one hook at a time, keep `rotate` and call `model(x, rotate_latent_index=k)` — `latents_by_source` will only contain the active hook for that step.